# Configuring Environment

In [3]:
# ! pip install pandas matplotlib plotly numpy


# Import statements

In [9]:
import os
import sqlite3
import pandas as pd

# Pull data in

In [10]:
Movies_Dataset = 'CSV-Files/The Movies Dataset'
TMDB_Dataset = 'CSV-Files/TMDB 5000 Movie Dataset'
datasets = [Movies_Dataset, TMDB_Dataset]
for dataset in datasets:
    print(f"Dataset: {dataset}")
    for file in os.listdir(dataset):
        if file.endswith(".csv"):
            print(file)
    print("\n")

Dataset: CSV-Files/The Movies Dataset
links_small.csv
links.csv
credits.csv
movies_metadata.csv
ratings.csv
ratings_small.csv
keywords.csv


Dataset: CSV-Files/TMDB 5000 Movie Dataset
tmdb_5000_credits.csv
tmdb_5000_movies.csv




# Creating PandasDataframes

In [11]:
Movies_Dataset = 'CSV-Files/The Movies Dataset'
TMDB_Dataset = 'CSV-Files/TMDB 5000 Movie Dataset'
datasets = [Movies_Dataset, TMDB_Dataset]
dataframes = {}

for dataset in datasets:
    print(f"Dataset: {dataset}")
    for file in os.listdir(dataset):
        if file.endswith(".csv"):
            dataframes[f"{dataset}/{file}"] = pd.read_csv(os.path.join(dataset, file))
            print(file)
    print("\n")

dataframes.keys()

Dataset: CSV-Files/The Movies Dataset
links_small.csv
links.csv
credits.csv


/var/folders/dz/d7pb5r1n2x9gw9xx4zjylysm0000gn/T/ipykernel_77490/1125748027.py:10: DtypeWarning: Columns (10: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframes[f"{dataset}/{file}"] = pd.read_csv(os.path.join(dataset, file))


movies_metadata.csv
ratings.csv
ratings_small.csv
keywords.csv


Dataset: CSV-Files/TMDB 5000 Movie Dataset
tmdb_5000_credits.csv
tmdb_5000_movies.csv




dict_keys(['CSV-Files/The Movies Dataset/links_small.csv', 'CSV-Files/The Movies Dataset/links.csv', 'CSV-Files/The Movies Dataset/credits.csv', 'CSV-Files/The Movies Dataset/movies_metadata.csv', 'CSV-Files/The Movies Dataset/ratings.csv', 'CSV-Files/The Movies Dataset/ratings_small.csv', 'CSV-Files/The Movies Dataset/keywords.csv', 'CSV-Files/TMDB 5000 Movie Dataset/tmdb_5000_credits.csv', 'CSV-Files/TMDB 5000 Movie Dataset/tmdb_5000_movies.csv'])

In [12]:
for dataset in dataframes.keys():
    print(dataset)
    print(dataframes[dataset].keys())

CSV-Files/The Movies Dataset/links_small.csv
Index(['movieId', 'imdbId', 'tmdbId'], dtype='str')
CSV-Files/The Movies Dataset/links.csv
Index(['movieId', 'imdbId', 'tmdbId'], dtype='str')
CSV-Files/The Movies Dataset/credits.csv
Index(['cast', 'crew', 'id'], dtype='str')
CSV-Files/The Movies Dataset/movies_metadata.csv
Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')
CSV-Files/The Movies Dataset/ratings.csv
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')
CSV-Files/The Movies Dataset/ratings_small.csv
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')
CSV-Files/The Movies Dataset/keywords.csv
Index(['id', 'keywo

# Convert to SQLite

In [15]:
# Save each dataset to its own database, with one table per CSV.
database_files = {
    Movies_Dataset: "movies_dataset.db",
    TMDB_Dataset: "tmdb_5000.db"
}

for dataset, database_file in database_files.items():
    connection = sqlite3.connect(database_file)
    try:
        for file_path, dataframe in dataframes.items():
            if os.path.dirname(file_path) == dataset:
                table_name = os.path.splitext(os.path.basename(file_path))[0].replace("-", "_")
                dataframe.to_sql(table_name, connection, if_exists="replace", index=False)
                print(f"{database_file}: {table_name} ({len(dataframe):,} rows)")
    finally:
        connection.close()


movies_dataset.db: links_small (9,125 rows)
movies_dataset.db: links (45,843 rows)
movies_dataset.db: credits (45,476 rows)
movies_dataset.db: movies_metadata (45,466 rows)
movies_dataset.db: ratings (26,024,289 rows)
movies_dataset.db: ratings_small (100,004 rows)
movies_dataset.db: keywords (46,419 rows)
tmdb_5000.db: tmdb_5000_credits (4,803 rows)
tmdb_5000.db: tmdb_5000_movies (4,803 rows)


In [16]:
# Write each database's table schemas to a matching Markdown file.
for database_file in database_files.values():
    connection = sqlite3.connect(f"file:{database_file}?mode=ro", uri=True)
    try:
        schemas = connection.execute(
            "SELECT name, sql FROM sqlite_master "
            "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
        markdown_file = os.path.splitext(database_file)[0] + ".md"
        with open(markdown_file, "w", encoding="utf-8") as file:
            file.write(f"# {database_file} schema\n\n")
            for table_name, sql in schemas:
                file.write(f"## {table_name}\n\n```sql\n{sql};\n```\n\n")
        print(f"Saved {markdown_file} ({len(schemas)} tables)")
    finally:
        connection.close()


Saved movies_dataset.md (7 tables)
Saved tmdb_5000.md (2 tables)


# Discover hidden tables inside stored text

Scan every row and column in both existing SQLite databases, decode JSON and Python literal containers, and write `hidden_tables.md` with expanded fields and counts. This section can run independently of the earlier import cells; it opens the databases read-only.


In [ ]:
from pathlib import Path
import ast
from collections import Counter
import json
import sqlite3


def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


def candidate_expression(name):
    column = quote_identifier(name)
    return (
        f"typeof({column}) = 'text' AND "
        f"substr(ltrim({column}, ' \t\r\n'), 1, 1) IN ('[', '{{')"
    )


def parse_container(text):
    # TMDB uses JSON; The Movies Dataset often uses Python repr (single quotes).
    for parser_name, parser in [('JSON', json.loads), ('Python literal', ast.literal_eval)]:
        try:
            value = parser(text.strip())
        except (ValueError, SyntaxError, TypeError, RecursionError):
            continue
        if isinstance(value, (list, dict)):
            return value, parser_name
    return None, None


def value_type(value):
    if value is None:
        return 'null'
    if isinstance(value, bool):
        return 'boolean'
    if isinstance(value, dict):
        return 'object'
    if isinstance(value, (list, tuple)):
        return 'array'
    if isinstance(value, int):
        return 'integer'
    if isinstance(value, float):
        return 'real'
    return 'string'


def inspect_container(value, profiles, path, seen):
    profile = profiles.setdefault(path, {
        'shapes': Counter(), 'occurrences': 0, 'parent_rows': 0,
        'empty': 0, 'records': 0, 'object_records': 0,
        'scalar_types': Counter(), 'fields': {},
    })
    seen.add(path)
    profile['occurrences'] += 1
    profile['shapes']['array' if isinstance(value, list) else 'object'] += 1
    profile['empty'] += int(not value)
    records = value if isinstance(value, list) else [value]
    for record in records:
        profile['records'] += 1
        if isinstance(record, dict):
            profile['object_records'] += 1
            for key, field_value in record.items():
                field = profile['fields'].setdefault(str(key), Counter())
                field[value_type(field_value)] += 1
                if isinstance(field_value, (list, dict)):
                    inspect_container(field_value, profiles, f'{path}.{key}', seen)
        else:
            profile['scalar_types'][value_type(record)] += 1
            if isinstance(record, list):
                inspect_container(record, profiles, path + '[]', seen)


def scan_databases(database_paths):
    results = []
    for database_path in database_paths:
        database_path = Path(database_path).resolve(strict=True)
        connection = sqlite3.connect(database_path.as_uri() + '?mode=ro', uri=True)
        try:
            tables = connection.execute(
                "SELECT name FROM sqlite_master WHERE type='table' "
                "AND name NOT LIKE 'sqlite_%' ORDER BY name"
            ).fetchall()
            for (table,) in tables:
                columns = [row[1] for row in connection.execute(
                    f'PRAGMA table_info({quote_identifier(table)})'
                )]
                # SQL checks EVERY column, including columns declared numeric.
                # Only bracket-prefixed text is transferred to Python for parsing.
                expressions = [
                    f'SUM(CASE WHEN {candidate_expression(column)} THEN 1 ELSE 0 END)'
                    for column in columns
                ]
                counts = connection.execute(
                    'SELECT COUNT(*), ' + ', '.join(expressions)
                    + f' FROM {quote_identifier(table)}'
                ).fetchone()
                table_result = {
                    'database': database_path.name, 'table': table,
                    'rows': counts[0], 'columns': columns, 'nested_columns': {},
                }
                for column, count in zip(columns, counts[1:]):
                    if not count:
                        continue
                    column_result = {
                        'candidates': count, 'parsers': Counter(), 'failed': 0,
                        'profiles': {},
                    }
                    cursor = connection.execute(
                        f'SELECT {quote_identifier(column)} FROM {quote_identifier(table)} '
                        f'WHERE {candidate_expression(column)}'
                    )
                    for (raw_value,) in cursor:
                        value, parser_name = parse_container(raw_value)
                        if parser_name is None:
                            column_result['failed'] += 1
                            continue
                        column_result['parsers'][parser_name] += 1
                        seen = set()
                        inspect_container(value, column_result['profiles'], '$', seen)
                        for path in seen:
                            column_result['profiles'][path]['parent_rows'] += 1
                    table_result['nested_columns'][column] = column_result
                results.append(table_result)
                print(f"Scanned {database_path.name}.{table}: {counts[0]:,} rows, {len(columns)} columns")
        finally:
            connection.close()
    return results


def write_hidden_tables_report(results, output_path):
    lines = [
        '# Hidden tables in the movie datasets', '',
        'Generated by the nested-data inspection section of `prototyping.ipynb`.', '',
        '## Method and scope', '',
        'Every row and column of every user table in both SQLite databases was checked. '
        'Text starting with `[` or `{` after whitespace was parsed first as JSON, '
        'then with `ast.literal_eval` as a Python literal. No `eval` is used. '
        'Lists and dictionaries are profiled recursively; field names and types are '
        'unions across all decoded records, not a sample. SQL declarations alone '
        'do not reveal this structure. The databases were opened read-only.', '',
        'Record counts are stored occurrences, including duplicates. Empty arrays '
        'contain zero records; a dictionary contributes one object record. Parent-row '
        'counts mean source rows with a decoded container, including empty containers. '
        'Blank/NULL cells and plain scalar text do not qualify. Other encodings '
        '(such as XML, delimited lists, or double-encoded JSON) are outside this detector.', '',
        '## All tables checked', '',
        '| Database | Table | Source rows | Columns checked | Container columns |',
        '|---|---|---:|---:|---|',
    ]
    for table in results:
        nested = [column for column, result in table['nested_columns'].items() if result['profiles']]
        lines.append(
            f"| {table['database']} | {table['table']} | {table['rows']:,} | "
            f"{len(table['columns'])} | {', '.join(nested) or 'None'} |"
        )
    lines += ['', '## Hidden-table summary', '',
              '| Database | Source column | Path | Shape | Parent rows | Records | Empty containers |',
              '|---|---|---|---|---:|---:|---:|']
    for table in results:
        for column, result in table['nested_columns'].items():
            for path, profile in sorted(result['profiles'].items()):
                lines.append(
                    f"| {table['database']} | {table['table']}.{column} | `{path}` | "
                    f"{', '.join(sorted(profile['shapes']))} | {profile['parent_rows']:,} | "
                    f"{profile['records']:,} | {profile['empty']:,} |"
                )
    lines += ['', '## Expanded fields', '',
              '`$` is the decoded source cell. Additional paths identify containers '
              'inside its records. Types describe observed values, not SQL constraints.', '']
    for table in results:
        for column, result in table['nested_columns'].items():
            lines += [f"### {table['database']} / {table['table']}.{column}", '',
                      f"Candidate cells: {result['candidates']:,}. "
                      f"Parsed: {sum(result['parsers'].values()):,}. "
                      f"Failed to decode as a list/object: {result['failed']:,}.", '',
                      'Formats: ' + (', '.join(f'{k}: {v:,}' for k, v in result['parsers'].items()) or 'None') + '.', '']
            for path, profile in sorted(result['profiles'].items()):
                lines += [f'Path `{path}`: {profile["object_records"]:,} object records.', '']
                if profile['fields']:
                    lines += ['| Field | Observed types | Present in object records | Missing from object records |',
                              '|---|---|---:|---:|']
                    for field, types in sorted(profile['fields'].items()):
                        present = sum(types.values())
                        lines.append(f"| {field} | {', '.join(sorted(types))} | {present:,} | {profile['object_records'] - present:,} |")
                    lines.append('')
                if profile['scalar_types']:
                    lines += ['Non-object elements: ' + ', '.join(
                        f'{k}: {v:,}' for k, v in profile['scalar_types'].items()
                    ) + '.', '']
    lines += ['## Relational interpretation', '',
              'An array of objects can become a child table with one row per element. '
              'Retain the source movie identifier and array position; cast/crew/keywords '
              'IDs identify the nested entities, so avoid confusing them with movie IDs. '
              'A single object such as `belongs_to_collection` can become a related '
              'collection table or flattened columns. Retain a source-row identifier '
              'when a movie ID is missing or duplicated. Countries and languages use '
              'their ISO codes. These are inferred structures; this report does not '
              'create tables or establish uniqueness/foreign-key constraints.', '']
    Path(output_path).write_text('\n'.join(lines), encoding='utf-8')


# This section runs independently of the earlier CSV import/database creation cells.
database_paths = [Path('movies_dataset.db'), Path('tmdb_5000.db')]
hidden_table_results = scan_databases(database_paths)
report_path = Path('hidden_tables.md')
write_hidden_tables_report(hidden_table_results, report_path)
hidden_column_count = sum(
    bool(result['profiles'])
    for table in hidden_table_results
    for result in table['nested_columns'].values()
)
print(f'Found {hidden_column_count} container columns. Saved {report_path.resolve()}')
